In [0]:
%python
# MAGIC
from datetime import datetime
import json

# Datos de ejemplo
#last_mod_dates = "{\"TodasFechasArchivos\": [{ \"name\": \"original_file_date\", \"value\": \"2025-05-30T19:06:01Z\" },{ \"name\": \"original_file_date\", \"value\": \"2025-05-31T05:25:01Z\" },{ \"name\": \"original_file_date\", \"value\": \"2025-03-28T19:50:02Z\" },{ \"name\": \"original_file_date\", \"value\": \"2025-03-26T04:15:01Z\" }]}"

# Ruta del archivo de última ingesta
#last_ingest_time = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/gestion_recursos/operacion_red/fatem/fallas_masivas_nuevas_react_fo_v2/last_ingest_time.txt"

# Definir widgets sin valores por defecto
dbutils.widgets.text("last_mod_dates", "", "Ruta last_mod_dates")
dbutils.widgets.text("last_ingest_time", "", "Ruta last_ingest_time")

# Obtener los valores de los widgets
last_mod_dates = dbutils.widgets.get("last_mod_dates")
last_ingest_time = dbutils.widgets.get("last_ingest_time")

# Convertir el JSON a objeto
json_data = json.loads(last_mod_dates)

# Extraer las fechas y convertirlas a datetime
fechas = [entry["value"] for entry in json_data["TodasFechasArchivos"]]
fechas_dt = [datetime.strptime(f, "%Y-%m-%dT%H:%M:%SZ") for f in fechas]
fecha_maxima = max(fechas_dt)
fecha_final = fecha_maxima.strftime("%Y-%m-%dT%H:%M:%S.000Z")
print("Fecha más reciente encontrada:", fecha_final)

# Leer la fecha actual del archivo (si existe)
try:
    current_content = dbutils.fs.head(last_ingest_time)
    current_lines = current_content.strip().split("\n")
    current_date_str = current_lines[1].replace(";", "").strip()
    current_date = datetime.strptime(current_date_str, "%Y-%m-%dT%H:%M:%S.%fZ")
    print(f"Fecha actual en el archivo: {current_date_str}")
except Exception as e:
    print(f"No se pudo leer la fecha actual del archivo. Se actualizará de todos modos. Detalle: {str(e)}")
    current_date = None

# Convertir la nueva fecha a datetime
new_date = datetime.strptime(fecha_final, "%Y-%m-%dT%H:%M:%S.%fZ")

# Verificar si es necesario actualizar
if current_date is None or new_date > current_date:
    try:
        new_content = f"timestamp;\n{fecha_final};"
        dbutils.fs.put(last_ingest_time, new_content, overwrite=True)
        print(f"Archivo {last_ingest_time} actualizado con la nueva fecha: {fecha_final}")
    except Exception as e:
        print(f"Error al actualizar el archivo {last_ingest_time}: {str(e)}")
        dbutils.notebook.exit("Error: No se pudo actualizar el archivo")
else:
    print(f"⏩ La fecha {fecha_final} NO es mayor que la actual. No se actualiza el archivo.")

# Leer nuevamente para mostrar el resultado final
try:
    content = dbutils.fs.head(last_ingest_time)
    print(f"📄 Contenido actual del archivo:\n{content}")
except Exception as e:
    print(f"Error al leer el archivo {last_ingest_time}: {str(e)}")


In [0]:
import org.apache.spark.sql.DataFrame
import java.time.LocalDate 
import java.time.format.DateTimeFormatter 
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import java.time.LocalDateTime

// Input parameters from widgets
var sourcePath = dbutils.widgets.get("sourcePath")
var targetPath = dbutils.widgets.get("targetPath")
val header = dbutils.widgets.get("header")
val delimiter = dbutils.widgets.get("delimiter")
val filePrefix = dbutils.widgets.get("filePrefix")
val name_folder = dbutils.widgets.get("name_folder")
val columnValidate = dbutils.widgets.get("columnValidate").split(",").map(_.trim).filter(_.nonEmpty)
val columnFormat = dbutils.widgets.get("columnFormat")
val columnNullOption = dbutils.widgets.get("columnNullOption").toBoolean

// Generate the file name with the current date (yyyyMMdd format)
val currentDateTime = LocalDateTime.now().format(DateTimeFormatter.ofPattern("yyyyMMddHHmm"))
val finalFileName = s"${filePrefix}_$name_folder.csv"
val finalPath = s"$targetPath$finalFileName"

// Function to delete the contents of the source folder
def deleteFolderContentsOnly(path: String): Unit = {
  try {
    val filesAndDirs = dbutils.fs.ls(path)
    if (filesAndDirs.nonEmpty) {
      filesAndDirs.foreach(file => dbutils.fs.rm(file.path, true))
      println(s"Se eliminó correctamente el contenido de la carpeta: $path")
    } else {
      println(s"La carpeta $path ya está vacía, no hay contenido que eliminar.")
    }
  } catch {
    case e: Exception => println(s"[ERROR] No se pudo eliminar el contenido de la carpeta $path: ${e.getMessage}")
  }
}

try {
  // Read all CSV files from the source folder
  val df: DataFrame = spark.read
    .option("header", header)
    .option("delimiter", delimiter)
    .option("inferSchema", "false")
    .csv(sourcePath + name_folder)

  // Verify if the DataFrame has data
  if (df.count() > 0) {
    df.printSchema()
    println(s"Número total de registros: ${df.count()}")

    // Verify that all specified columns exist
    columnValidate.foreach { colName =>
      if (!df.columns.contains(colName)) {
        throw new IllegalArgumentException(s"La columna $colName no existe en el DataFrame")
      }
    }

    // Create the filter condition for all columns
    val filterCondition = columnValidate.map { colName =>
      if (columnNullOption) {
        // If nullOption is true, accept nulls and validate format for non-nulls
        col(colName).isNull || to_timestamp(col(colName), columnFormat).isNotNull
      } else {
        // If nullOption is false, do not accept nulls and validate format
        col(colName).isNotNull && to_timestamp(col(colName), columnFormat).isNotNull
      }
    }.reduceOption(_ && _).getOrElse(lit(true)) // Combine with AND, use true if no columns

    // Filter records where ALL columns meet the format
    val filteredDf = df.filter(filterCondition)

    // Verify if the filtered DataFrame has data
    if (!columnNullOption && filteredDf.count() == 0) {
      // Identify columns with all null values
      val nullColumns = columnValidate.filter { colName =>
        df.filter(col(colName).isNotNull).count() == 0
      }
      // If there are null columns, throw an exception with details
      if (nullColumns.nonEmpty) {
        throw new IllegalStateException(s"Error: No hay registros válidos. Las columnas ${nullColumns.mkString(", ")} tienen todos sus valores nulos, lo cual no está permitido cuando columnNullOption es false.")
      } else {
        throw new IllegalStateException(s"Error: No hay registros válidos. Las columnas ${columnValidate.mkString(", ")} no cumplen con el formato requerido (${columnFormat}) cuando columnNullOption es false.")
      }
    }

    // Write to a temporary folder
    val tempPath = s"$targetPath/temp"
    filteredDf.repartition(1)
      .write
      .mode("overwrite")
      .option("delimiter", delimiter)
      .option("header", header)
      .csv(tempPath)

    // Get the name of the file generated by Spark
    val files = dbutils.fs.ls(tempPath).filter(_.name.startsWith("part-")).map(_.path)
    val generatedFile = files.head

    // Rename to finalPath
    dbutils.fs.mv(generatedFile, finalPath)

    // Delete the temporary folder
    dbutils.fs.rm(tempPath, true)

    println(s"Archivo combinado guardado en: $finalPath")
    
    // Delete the contents of the source folder
    deleteFolderContentsOnly(sourcePath)

    // Notebook output for successful processing
    dbutils.notebook.exit(finalPath)
  } else {
    // Handle the case where the DataFrame is empty
    println("No se encontraron datos en los archivos CSV de la ruta de origen.")
    // Delete the contents of the source folder
    deleteFolderContentsOnly(sourcePath)
    // Exit with a specific output for ADF
    dbutils.notebook.exit("NO_DATA_FOUND")
  }
} catch {
  case e: Exception =>
    println(s"Error al procesar los archivos: ${e.getMessage}")
    println("No se generó ningún archivo de salida debido a que no hay datos o ocurrió un problema.")
    // Delete the contents of the source folder even in case of an error
    deleteFolderContentsOnly(sourcePath)
    // Exit with an error message for ADF
    dbutils.notebook.exit(s"ERROR: ${e.getMessage}")
}